## Demo Debezium

Поднимаем стенд
```bash
docker-compose up -d
```

Debezium UI: http://localhost:8080<br>
SQL выполняем прямо отсюда, через функцию `sql()`. Постгрес слушает на `localhost:5432`

In [1]:
import json
from time import sleep

import psycopg2
import requests
from kafka import KafkaAdminClient, KafkaConsumer

KAFKA_HOST = "localhost"
KAFKA_CONSUMER_GROUP = "backend"
CONNECT_URL = "http://localhost:8083/connectors"

conn = psycopg2.connect(host="localhost", port=5432, user="postgres", password="postgres", dbname="postgres")
conn.autocommit = True


def sql(query):
    # Выполняем запрос; если он что-то возвращает (select), отдаем строки
    with conn.cursor() as cur:
        cur.execute(query)
        if cur.description:
            return cur.fetchall()

### Шаг 1

На БД `postgres` выполняем

In [2]:
sql("select * from pg_stat_replication;")

[]

Видим, что ответ пустой; Значит, сейчас репликации нет;

Далее создаем таблицу и заливаем в нее данные

In [3]:
sql("""
create table public.customers (
    id int,
    name text
);

alter table public.customers replica identity full;

insert into public.customers (id, name)
values
    (1, 'ilya'), (2, 'kate'), (3, 'ivan')
;
""")

Это будут наши стартовые значения

### Шаг 2

Настройка debezium

In [4]:
connector = {
  "name": "pg-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "plugin.name": "pgoutput",

    "database.hostname": "postgres",
    "database.port": "5432",
    "database.user": "postgres",
    "database.password": "postgres",
    "database.dbname" : "postgres",

    "table.include.list": "public.(.*)",
    "heartbeat.interval.ms": "5000",
    "slot.name": "dbname_debezium",
    "publication.name": "dbname_publication",

    "topic.creation.default.cleanup.policy": "delete",
    "topic.creation.default.partitions": "1",
    "topic.creation.default.replication.factor": "1",
    "topic.creation.default.retention.ms": "604800000",
    "topic.creation.enable": "true",
    "topic.prefix": "postgres"
  }
}

In [5]:
# REST API Kafka Connect. PUT создает коннектор, а если он уже есть - обновляет,
# поэтому ячейку можно безопасно запускать повторно
response = requests.put(f"{CONNECT_URL}/{connector['name']}/config", json=connector["config"])
assert response.status_code < 400, response.text
response.status_code

201

Подключение создалось;<br>
Чтобы дополнительно в этом убедиться, можем посмотреть в UI: http://localhost:8080<br>
Или спросить у Kafka Connect:

In [6]:
sleep(5)  # даем коннектору запуститься

status = requests.get(f"{CONNECT_URL}/{connector['name']}/status").json()
print(status["connector"]["state"], [task["state"] for task in status["tasks"]])

RUNNING ['RUNNING']


Теперь в `pg_stat_replication` появился Debezium:

In [7]:
sql("select application_name, state from pg_stat_replication;")

[('Debezium Streaming', 'streaming')]

### Шаг 3

Смотрим, что топик в kafka создался

In [8]:
admin = KafkaAdminClient(bootstrap_servers=f"{KAFKA_HOST}:9092")


def show_topics():
    for topic in sorted(admin.list_topics()):
        if topic != "__consumer_offsets":
            print(topic)


show_topics()

__debezium-heartbeat.postgres
connect_configs
connect_offsets
connect_statuses
postgres.public.customers
service_backend_feed


### Шаг 3

Видим, что при init для таблицы, в которой у нас уже были строки, уже создался топик<br>
Давайте посмотрим, что туда записалось

In [9]:
def read_single_message(topic=None):
    assert topic is not None, 'You must specify topic name'

    consumer = KafkaConsumer(
        bootstrap_servers=f"{KAFKA_HOST}:9092",
        value_deserializer=lambda v: v if v is None else json.loads( v.decode("utf-8") ),
        auto_offset_reset="earliest",
        group_id=KAFKA_CONSUMER_GROUP
    )
    consumer.subscribe(topics=topic)

    try:
        for message in consumer:
            value = message.value
            return value
    except Exception as e:
            print("Closing consumer due to error\n")
            consumer.close()
            raise e
    finally:
        print("Closing consumer due to finish\n")
        consumer.close()

In [10]:
value = read_single_message(['postgres.public.customers'])
value

Closing consumer due to finish



{'schema': {'type': 'struct',
  'fields': [{'type': 'struct',
    'fields': [{'type': 'int32', 'optional': True, 'field': 'id'},
     {'type': 'string', 'optional': True, 'field': 'name'}],
    'optional': True,
    'name': 'postgres.public.customers.Value',
    'field': 'before'},
   {'type': 'struct',
    'fields': [{'type': 'int32', 'optional': True, 'field': 'id'},
     {'type': 'string', 'optional': True, 'field': 'name'}],
    'optional': True,
    'name': 'postgres.public.customers.Value',
    'field': 'after'},
   {'type': 'struct',
    'fields': [{'type': 'string', 'optional': False, 'field': 'version'},
     {'type': 'string', 'optional': False, 'field': 'connector'},
     {'type': 'string', 'optional': False, 'field': 'name'},
     {'type': 'int64', 'optional': False, 'field': 'ts_ms'},
     {'type': 'string',
      'optional': True,
      'name': 'io.debezium.data.Enum',
      'version': 1,
      'parameters': {'allowed': 'true,first,first_in_data_collection,last_in_data_co

Что у нас тут получилось:
1) Есть схема сообщения
2) Так как все строки новые, то `before` содержит `None`
3) В то же время, `after` содержит значения в строке

Посмотрим, что происходит при insert

In [11]:
sql("""
insert into public.customers (id, name)
values
    (4, 'david')
;
""")

In [12]:
# Две строки из стартового набора мы еще не читали, пропускаем их
for i in range(2):
    read_single_message(['postgres.public.customers'])

Closing consumer due to finish

Closing consumer due to finish



In [13]:
value = read_single_message(['postgres.public.customers'])
value['payload']

Closing consumer due to finish



{'before': None,
 'after': {'id': 4, 'name': 'david'},
 'source': {'version': '3.3.1.Final',
  'connector': 'postgresql',
  'name': 'postgres',
  'ts_ms': 1792501930791,
  'snapshot': 'false',
  'db': 'postgres',
  'sequence': '[null,"23029400"]',
  'ts_us': 1792501930791512,
  'ts_ns': 1792501930791512000,
  'schema': 'public',
  'table': 'customers',
  'txId': 489,
  'lsn': 23029400,
  'xmin': None},
 'transaction': None,
 'op': 'c',
 'ts_ms': 1792501931059,
 'ts_us': 1792501931059327,
 'ts_ns': 1792501931059327595}

Получили все то же самое<br>
Теперь посмотрим на update

In [14]:
sql("""
update public.customers
set name = 'diana'
where id = 4;
""")

In [15]:
value = read_single_message(['postgres.public.customers'])
value['payload']

Closing consumer due to finish



{'before': {'id': 4, 'name': 'david'},
 'after': {'id': 4, 'name': 'diana'},
 'source': {'version': '3.3.1.Final',
  'connector': 'postgresql',
  'name': 'postgres',
  'ts_ms': 1792501980109,
  'snapshot': 'false',
  'db': 'postgres',
  'sequence': '["23029520","23029576"]',
  'ts_us': 1792501980109827,
  'ts_ns': 1792501980109827000,
  'schema': 'public',
  'table': 'customers',
  'txId': 490,
  'lsn': 23029576,
  'xmin': None},
 'transaction': None,
 'op': 'u',
 'ts_ms': 1792501980467,
 'ts_us': 1792501980467218,
 'ts_ns': 1792501980467218215}

Что мы имеем:
1) `before` содержит старые значения в строке
2) `after` содержит новые значения в строке

Посмотрим, что произойдет при delete

In [16]:
sql("""
delete
from public.customers
where id = 4;
""")

In [17]:
value = read_single_message(['postgres.public.customers'])
value['payload']

Closing consumer due to finish



{'before': {'id': 4, 'name': 'diana'},
 'after': None,
 'source': {'version': '3.3.1.Final',
  'connector': 'postgresql',
  'name': 'postgres',
  'ts_ms': 1792501994823,
  'snapshot': 'false',
  'db': 'postgres',
  'sequence': '["23029720","23029776"]',
  'ts_us': 1792501994823917,
  'ts_ns': 1792501994823917000,
  'schema': 'public',
  'table': 'customers',
  'txId': 491,
  'lsn': 23029776,
  'xmin': None},
 'transaction': None,
 'op': 'd',
 'ts_ms': 1792501994881,
 'ts_us': 1792501994881788,
 'ts_ns': 1792501994881788041}

Что мы имеем:
1) `before` содержит старые значения в строке
2) `after` содержит `None`

<b>ВАЖНО:</b> Debezium [не умеет](https://github.com/MaterializeInc/materialize/issues/6596#issuecomment-963479859) в обработку truncate

### Шаг 4

Посмотрим, что произойдет при создании новой таблицы

In [18]:
sql("""
create table public.stores (
    id int,
    city text
);
alter table public.stores replica identity full;
""")

In [19]:
show_topics()

__debezium-heartbeat.postgres
connect_configs
connect_offsets
connect_statuses
postgres.public.customers
service_backend_feed


Видим, что топика нет<br>
Все потому, что таблица пустая<br>
Добавим в нее строк

In [20]:
sql("""
insert into public.stores (id, city)
values
    (1, 'Moscow'),
    (2, 'London'),
    (3, 'Milano')
;
""")
sleep(3)  # даем Debezium создать топик

In [21]:
show_topics()

__debezium-heartbeat.postgres
connect_configs
connect_offsets
connect_statuses
postgres.public.customers
postgres.public.stores
service_backend_feed


Теперь топик есть, и его можно почитать

In [22]:
for i in range(3):
    value = read_single_message(['postgres.public.stores'])
    print(value['payload'], end='\n---\n')

Closing consumer due to finish

{'before': None, 'after': {'id': 1, 'city': 'Moscow'}, 'source': {'version': '3.3.1.Final', 'connector': 'postgresql', 'name': 'postgres', 'ts_ms': 1792502025075, 'snapshot': 'false', 'db': 'postgres', 'sequence': '["23066856","23066984"]', 'ts_us': 1792502025075925, 'ts_ns': 1792502025075925000, 'schema': 'public', 'table': 'stores', 'txId': 493, 'lsn': 23066984, 'xmin': None}, 'transaction': None, 'op': 'c', 'ts_ms': 1792502025268, 'ts_us': 1792502025268523, 'ts_ns': 1792502025268523083}
---
Closing consumer due to finish

{'before': None, 'after': {'id': 2, 'city': 'London'}, 'source': {'version': '3.3.1.Final', 'connector': 'postgresql', 'name': 'postgres', 'ts_ms': 1792502025075, 'snapshot': 'false', 'db': 'postgres', 'sequence': '["23066856","23067056"]', 'ts_us': 1792502025075925, 'ts_ns': 1792502025075925000, 'schema': 'public', 'table': 'stores', 'txId': 493, 'lsn': 23067056, 'xmin': None}, 'transaction': None, 'op': 'c', 'ts_ms': 1792502025268,